# T01 — Data-engineering decision evidence

This notebook gathers development-only evidence for Issue #2. It does **not** implement the production data pipeline, choose a resource threshold, promote DuckDB/Polars, choose a Parquet codec, train a model, construct the frozen split, or access held-out evaluation.

Authority: `docs/decision_register.csv`, numbered contracts, and `docs/adr/ADR-T01-data-engineering.md`. The raw compressed artifact and decompressed CSV are already checksum-reconciled. This notebook additionally checks the current CSV-to-Parquet relationship before using that Parquet as benchmark input.

## Predeclared benchmark and controlled-confirmation protocol

### Common controls

- Development machine only; record OS, CPU count, total RAM, Python, Pandas, PyArrow and psutil versions.
- Use the current Pandas + PyArrow default path. No DuckDB/Polars comparison or promotion.
- Use the checksum-identified raw CSV and the existing full Parquet derivative. Before benchmarking, parse the CSV in Pandas `float64`/`int64` chunks and compare every value against the corresponding Parquet value in source order. The parser configuration is part of the evidence because different valid decimal-to-binary parsers can differ by one ULP.
- `_source_row_id` is the zero-based canonical source-row ordinal, appended for benchmark identity/order checks and never used as a feature.
- Record raw measurements. No post-hoc RAM PASS/FAIL threshold and no automatic compression winner.
- Results are written beneath one new `outputs/runs/<run_id>/audit/` development run. The notebook creates a new run ID on every execution.
- This execution mode is a clean-kernel, full-data-only D04 confirmation. It does not repeat the earlier scale ladder or D06 codec comparison.

### D04 resource protocol — decision remains pending

The earlier decision-evidence run followed the frozen nested scale progression `50K → 500K → 2M → full`. This controlled confirmation executes only the full rung in a fresh kernel. It scans the deterministic source with PyArrow, appends the original source ordinal, and explicitly materializes to Pandas. It records process RSS separately from starting/minimum system-available memory and pagefile/swap counters, plus wall-clock time, row count, Arrow schema, Pandas dtypes, order and source identity.

A rung records `COMPLETED` or the actual exception/resource failure. If a rung fails, later rungs are not attempted. Swap observations are reported, not converted into an invented decision threshold.

### D06 compression protocol — decision remains pending

The earlier decision-evidence run compared exactly A=`snappy` and B=`zstd` under the predeclared controls below. The controlled D04 confirmation does not execute D06 again; the code remains visible for auditability and D06 remains owner-pending.

For each repetition, measure write time/RSS, end-to-end Parquet→Arrow→Pandas read time/RSS, and file size. Verify exact row count, schema, Pandas dtypes, `_source_row_id` order and full Arrow semantic equality. Report per-run values and medians/variability. Temporary candidate files are deleted only after their hash and verification record are captured. No score or winner is computed.

In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import itertools
import os
import platform
import shutil
import statistics
import subprocess
import sys
import threading
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import pyarrow as pa
import pyarrow.dataset as pads
import pyarrow.parquet as pq
from IPython.display import Markdown, display

EXPECTED_ROWS = 13_979_592
FEATURE_COLUMNS = [f"f{i}" for i in range(12)]
VALUE_COLUMNS = FEATURE_COLUMNS + ["treatment", "conversion", "visit", "exposure"]
IDENTITY_COLUMN = "_source_row_id"
EXECUTION_MODE = "D04_FULL_CONFIRMATION_ONLY"
SCALES = [EXPECTED_ROWS]
ROW_GROUP_SIZE = 1_048_576
REPETITIONS = 0
ARROW_TYPES = {**{c: pa.float64() for c in FEATURE_COLUMNS}, **{c: pa.int64() for c in VALUE_COLUMNS[12:]}}

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "docs" / "decision_register.csv").is_file():
            return candidate
    raise RuntimeError("Repository root not found")

REPO_ROOT = find_repo_root(Path.cwd())
RAW_CSV = REPO_ROOT / "data" / "raw" / "criteo-uplift-v2.1.csv"
SOURCE_PARQUET = REPO_ROOT / "data" / "processed" / "criteo-uplift-v2.1.parquet"
for required in (RAW_CSV, SOURCE_PARQUET):
    if not required.is_file():
        raise FileNotFoundError(required)

run_id = "t01_d04_confirmation_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_%f")
RUN_ROOT = REPO_ROOT / "outputs" / "runs" / run_id
AUDIT_DIR = RUN_ROOT / "audit"
TEMP_DIR = RUN_ROOT / "temporary_candidates"
AUDIT_DIR.mkdir(parents=True, exist_ok=False)
TEMP_DIR.mkdir(parents=True, exist_ok=False)

def sha256_file(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(path: Path, payload) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

def git_value(*args: str) -> str | None:
    try:
        return subprocess.check_output(["git", *args], cwd=REPO_ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None

class ResourceMonitor:
    def __init__(self, interval_seconds: float = 0.02):
        self.interval_seconds = interval_seconds
        self.process = psutil.Process(os.getpid())
        self.stop_event = threading.Event()
        self.thread = None

    def _sample(self):
        while not self.stop_event.is_set():
            rss = self.process.memory_info().rss
            vm = psutil.virtual_memory()
            swap = psutil.swap_memory()
            self.peak_rss = max(self.peak_rss, rss)
            self.min_available = min(self.min_available, vm.available)
            self.peak_swap_used = max(self.peak_swap_used, swap.used)
            self.peak_swap_sin = max(self.peak_swap_sin, getattr(swap, "sin", 0))
            self.peak_swap_sout = max(self.peak_swap_sout, getattr(swap, "sout", 0))
            self.stop_event.wait(self.interval_seconds)

    def __enter__(self):
        vm = psutil.virtual_memory()
        swap = psutil.swap_memory()
        self.baseline_rss = self.process.memory_info().rss
        self.baseline_system_total = vm.total
        self.baseline_system_available = vm.available
        self.baseline_system_used = vm.used
        self.baseline_system_percent = vm.percent
        self.baseline_swap_total = swap.total
        self.baseline_swap_free = swap.free
        self.peak_rss = self.baseline_rss
        self.min_available = vm.available
        self.baseline_swap_used = swap.used
        self.peak_swap_used = swap.used
        self.baseline_swap_sin = getattr(swap, "sin", 0)
        self.baseline_swap_sout = getattr(swap, "sout", 0)
        self.peak_swap_sin = self.baseline_swap_sin
        self.peak_swap_sout = self.baseline_swap_sout
        self.started = time.perf_counter()
        self.thread = threading.Thread(target=self._sample, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed_seconds = time.perf_counter() - self.started
        self.stop_event.set()
        self.thread.join(timeout=2)
        self.final_rss = self.process.memory_info().rss
        final_vm = psutil.virtual_memory()
        final_swap = psutil.swap_memory()
        self.final_system_available = final_vm.available
        self.final_system_used = final_vm.used
        self.final_swap_used = final_swap.used
        self.final_swap_free = final_swap.free
        self.min_available = min(self.min_available, final_vm.available)
        self.peak_swap_used = max(self.peak_swap_used, final_swap.used)

    def record(self):
        return {
            "wall_seconds": self.elapsed_seconds,
            "baseline_rss_bytes": self.baseline_rss,
            "peak_rss_bytes": self.peak_rss,
            "peak_rss_delta_bytes": max(0, self.peak_rss - self.baseline_rss),
            "system_total_bytes": self.baseline_system_total,
            "starting_system_available_bytes": self.baseline_system_available,
            "minimum_system_available_bytes": self.min_available,
            "final_system_available_bytes": self.final_system_available,
            "starting_system_used_bytes": self.baseline_system_used,
            "starting_system_memory_percent": self.baseline_system_percent,
            "final_system_used_bytes": self.final_system_used,
            "swap_total_bytes": self.baseline_swap_total,
            "starting_swap_free_bytes": self.baseline_swap_free,
            "baseline_swap_used_bytes": self.baseline_swap_used,
            "peak_swap_used_bytes": self.peak_swap_used,
            "swap_used_increase_bytes": max(0, self.peak_swap_used - self.baseline_swap_used),
            "final_swap_used_bytes": self.final_swap_used,
            "final_swap_free_bytes": self.final_swap_free,
            "swap_in_delta_bytes": max(0, self.peak_swap_sin - self.baseline_swap_sin),
            "swap_out_delta_bytes": max(0, self.peak_swap_sout - self.baseline_swap_sout),
        }

environment = {
    "run_id": run_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "logical_cpu_count": psutil.cpu_count(logical=True),
    "physical_cpu_count": psutil.cpu_count(logical=False),
    "cpu_frequency_mhz": getattr(psutil.cpu_freq(), "current", None),
    "cpu_percent_before_run": psutil.cpu_percent(interval=0.2),
    "total_ram_bytes": psutil.virtual_memory().total,
    "python": sys.version,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "pyarrow": pa.__version__,
    "psutil": psutil.__version__,
    "git_head": git_value("rev-parse", "HEAD"),
    "git_status_porcelain": git_value("status", "--porcelain=v1"),
}
write_json(AUDIT_DIR / "environment.json", environment)
display(Markdown(f"**Run ID:** `{run_id}`  \n**RAM:** {environment['total_ram_bytes'] / 2**30:.2f} GiB  \n**PyArrow:** {pa.__version__}; **Pandas:** {pd.__version__}"))

**Run ID:** `t01_d04_confirmation_20260812T043244Z_565265`  
**RAM:** 15.50 GiB  
**PyArrow:** 21.0.0; **Pandas:** 2.3.3

## Source reconciliation gate

The earlier successful streaming reconciliation compared every parsed `float64`/`int64` value at the same source-row ordinal. To keep this full-memory confirmation isolated, this run recomputes both file SHA-256 values and requires them to match that immutable-style reconciliation artifact and its own SHA-256. This preserves exact data identity without retaining the reconciliation allocation in the process being measured.

In [2]:
REFERENCE_RECONCILIATION = REPO_ROOT / "outputs/runs/t01_benchmark_20260812T040838Z_649715/audit/t01_source_reconciliation.json"
EXPECTED_RECONCILIATION_SHA256 = "084c1483872ce567848b28650c529fa8251c639897a6eaf11f8b3b9be3ee3303"
if not REFERENCE_RECONCILIATION.is_file():
    raise FileNotFoundError(REFERENCE_RECONCILIATION)
if sha256_file(REFERENCE_RECONCILIATION) != EXPECTED_RECONCILIATION_SHA256:
    raise RuntimeError("Pinned source-reconciliation artifact hash mismatch")
reference_reconciliation = json.loads(REFERENCE_RECONCILIATION.read_text(encoding="utf-8"))
raw_hash = sha256_file(RAW_CSV)
parquet_hash = sha256_file(SOURCE_PARQUET)
identity_valid = (
    reference_reconciliation.get("semantic_identity") == "PASS"
    and reference_reconciliation.get("rows_compared") == EXPECTED_ROWS
    and reference_reconciliation.get("raw_csv_sha256") == raw_hash
    and reference_reconciliation.get("source_parquet_sha256") == parquet_hash
)
source_reconciliation = {
    "run_id": run_id,
    "mode": "TRANSITIVE_IDENTITY_FROM_PINNED_FULL_RECONCILIATION",
    "reference_artifact": REFERENCE_RECONCILIATION.relative_to(REPO_ROOT).as_posix(),
    "reference_artifact_sha256": EXPECTED_RECONCILIATION_SHA256,
    "raw_csv_sha256": raw_hash,
    "source_parquet_sha256": parquet_hash,
    "rows_compared_in_reference": reference_reconciliation.get("rows_compared"),
    "columns_compared_in_reference": reference_reconciliation.get("columns_compared"),
    "csv_parser_in_reference": reference_reconciliation.get("csv_parser"),
    "semantic_digest_algorithm": reference_reconciliation.get("semantic_digest_algorithm"),
    "raw_semantic_digest": reference_reconciliation.get("raw_semantic_digest"),
    "parquet_semantic_digest": reference_reconciliation.get("parquet_semantic_digest"),
    "semantic_identity": "PASS" if identity_valid else "FAIL",
}
write_json(AUDIT_DIR / "t01_source_reconciliation_reference.json", source_reconciliation)
if not identity_valid:
    raise RuntimeError(f"Current source identities do not match pinned full reconciliation: {source_reconciliation}")
display(Markdown(f"**Current source identity:** PASS by checksum match to pinned full reconciliation over **{EXPECTED_ROWS:,}** ordered rows."))
del reference_reconciliation
gc.collect()

**Current source identity:** PASS by checksum match to pinned full reconciliation over **13,979,592** ordered rows.

52

## D04 execution — Pandas + PyArrow resource observations

These measurements describe the current machine. `COMPLETED` is an execution outcome, not an owner-approved resource PASS threshold.

In [3]:
dataset = pads.dataset(SOURCE_PARQUET, format="parquet")
d04_results = []
expected_arrow_schema = pa.schema([pa.field(c, ARROW_TYPES[c]) for c in VALUE_COLUMNS])
expected_pandas_dtypes = {**{c: "float64" for c in FEATURE_COLUMNS}, **{c: "int64" for c in VALUE_COLUMNS[12:]}, IDENTITY_COLUMN: "int64"}

for requested_rows in SCALES:
    gc.collect()
    result = {"requested_rows": requested_rows, "status": "NOT_RUN"}
    table = frame = None
    try:
        with ResourceMonitor() as monitor:
            scanner = dataset.scanner(columns=VALUE_COLUMNS, batch_size=131_072, use_threads=True)
            table = scanner.to_table() if requested_rows == EXPECTED_ROWS else scanner.head(requested_rows)
            source_ids = pa.array(np.arange(table.num_rows, dtype=np.int64))
            table = table.append_column(IDENTITY_COLUMN, source_ids)
            frame = table.to_pandas(split_blocks=True, self_destruct=False)
        result.update(monitor.record())
        result.update({
            "status": "COMPLETED",
            "observed_rows": len(frame),
            "observed_columns": list(frame.columns),
            "arrow_schema": str(table.schema),
            "pandas_dtypes": {c: str(t) for c, t in frame.dtypes.items()},
            "row_count_valid": len(frame) == requested_rows,
            "value_schema_valid": table.select(VALUE_COLUMNS).schema.equals(expected_arrow_schema),
            "pandas_dtypes_valid": {c: str(t) for c, t in frame.dtypes.items()} == expected_pandas_dtypes,
            "source_id_order_valid": frame[IDENTITY_COLUMN].iloc[0] == 0 and frame[IDENTITY_COLUMN].iloc[-1] == requested_rows - 1 and frame[IDENTITY_COLUMN].is_monotonic_increasing,
            "swap_io_observed": monitor.record()["swap_in_delta_bytes"] > 0 or monitor.record()["swap_out_delta_bytes"] > 0,
        })
    except (MemoryError, pa.ArrowMemoryError) as exc:
        result.update({"status": "RESOURCE_FAILURE", "exception_type": type(exc).__name__, "exception": str(exc)})
    except Exception as exc:
        result.update({"status": "EXECUTION_FAILURE", "exception_type": type(exc).__name__, "exception": str(exc)})
    finally:
        del table, frame
        gc.collect()
    d04_results.append(result)
    if result["status"] != "COMPLETED":
        break

d04_payload = {
    "run_id": run_id,
    "decision": "T01-D04",
    "decision_status": "PENDING_EMPIRICAL_EVIDENCE",
    "threshold_selected": False,
    "execution_mode": EXECUTION_MODE,
    "exact_operation": "PyArrow Dataset Scanner.to_table over all VALUE_COLUMNS; append zero-based canonical source ordinal; explicit Arrow-to-Pandas materialization",
    "data_identity": {"raw_csv_sha256": raw_hash, "source_parquet_sha256": parquet_hash, "semantic_digest": source_reconciliation["parquet_semantic_digest"], "source_reconciliation_status": source_reconciliation["semantic_identity"]},
    "engine": "Pandas + PyArrow",
    "scale_progression": SCALES,
    "selection": "deterministic canonical source prefix; nested engineering benchmark only",
    "results": d04_results,
}
write_json(AUDIT_DIR / "t01_d04_full_memory_confirmation.json", d04_payload)
pd.DataFrame(d04_results)[["requested_rows", "status", "wall_seconds", "peak_rss_bytes", "peak_rss_delta_bytes", "minimum_system_available_bytes", "swap_used_increase_bytes", "swap_io_observed"]]

,requested_rows,status,wall_seconds,peak_rss_bytes,peak_rss_delta_bytes,minimum_system_available_bytes,swap_used_increase_bytes,swap_io_observed
0,13979592,COMPLETED,4.677462,2359005184,2204852224,176128,1207156736,False


## D06 execution — bounded Snappy versus ZSTD evidence

The setup materializes one full Arrow source table and appends canonical source ordinals outside codec write/read timings. Both candidates receive the same table and writer settings.

In [4]:
if not d04_results or d04_results[-1].get("requested_rows") != EXPECTED_ROWS or d04_results[-1].get("status") != "COMPLETED":
    raise RuntimeError("D06 full-data benchmark not attempted because the D04 full rung did not complete")

gc.collect()
source_table = None
if REPETITIONS > 0:
    source_table = dataset.scanner(columns=VALUE_COLUMNS, batch_size=131_072, use_threads=True).to_table()
    source_table = source_table.append_column(IDENTITY_COLUMN, pa.array(np.arange(source_table.num_rows, dtype=np.int64)))
    source_schema = source_table.schema
else:
    source_schema = expected_arrow_schema.append(pa.field(IDENTITY_COLUMN, pa.int64()))
source_dtype_map = expected_pandas_dtypes
d06_results = []
writer_settings = {
    "row_group_size": ROW_GROUP_SIZE,
    "use_dictionary": True,
    "write_statistics": True,
    "data_page_version": "1.0",
    "version": "2.6",
}

for codec in ("snappy", "zstd"):
    for repetition in range(1, REPETITIONS + 1):
        candidate_path = TEMP_DIR / f"candidate_{codec}_rep{repetition}.parquet"
        record = {"codec": codec, "repetition": repetition, "status": "NOT_RUN"}
        candidate_table = candidate_frame = None
        try:
            gc.collect()
            with ResourceMonitor() as write_monitor:
                pq.write_table(source_table, candidate_path, compression=codec, **writer_settings)
            record["write"] = write_monitor.record()
            record["file_size_bytes"] = candidate_path.stat().st_size
            record["file_sha256"] = sha256_file(candidate_path)

            gc.collect()
            with ResourceMonitor() as read_monitor:
                candidate_table = pq.read_table(candidate_path, use_threads=True)
                candidate_frame = candidate_table.to_pandas(split_blocks=True, self_destruct=False)
            record["read"] = read_monitor.record()
            observed_dtypes = {c: str(t) for c, t in candidate_frame.dtypes.items()}
            record.update({
                "status": "COMPLETED",
                "row_count": candidate_table.num_rows,
                "row_count_valid": candidate_table.num_rows == EXPECTED_ROWS,
                "schema_valid": candidate_table.schema.equals(source_schema, check_metadata=False),
                "pandas_dtypes": observed_dtypes,
                "pandas_dtypes_valid": observed_dtypes == source_dtype_map,
                "source_id_order_valid": candidate_table[IDENTITY_COLUMN].equals(source_table[IDENTITY_COLUMN]),
                "semantic_identity": candidate_table.equals(source_table, check_metadata=False),
            })
            verification_fields = ["row_count_valid", "schema_valid", "pandas_dtypes_valid", "source_id_order_valid", "semantic_identity"]
            record["verification_pass"] = all(record[field] for field in verification_fields)
            if not record["verification_pass"]:
                record["status"] = "VERIFICATION_FAILURE"
        except (MemoryError, pa.ArrowMemoryError) as exc:
            record.update({"status": "RESOURCE_FAILURE", "exception_type": type(exc).__name__, "exception": str(exc)})
        except Exception as exc:
            record.update({"status": "EXECUTION_FAILURE", "exception_type": type(exc).__name__, "exception": str(exc)})
        finally:
            del candidate_table, candidate_frame
            gc.collect()
            if candidate_path.exists():
                candidate_path.unlink()
        d06_results.append(record)

del source_table
gc.collect()

def summarize_codec(codec: str):
    completed = [r for r in d06_results if r["codec"] == codec and r["status"] == "COMPLETED"]
    if not completed:
        return {"codec": codec, "completed_repetitions": 0}
    def values(path):
        result = []
        for row in completed:
            value = row
            for part in path:
                value = value[part]
            result.append(value)
        return result
    summary = {"codec": codec, "completed_repetitions": len(completed)}
    for label, path in {
        "write_seconds": ("write", "wall_seconds"),
        "read_seconds": ("read", "wall_seconds"),
        "write_peak_rss_bytes": ("write", "peak_rss_bytes"),
        "read_peak_rss_bytes": ("read", "peak_rss_bytes"),
        "file_size_bytes": ("file_size_bytes",),
    }.items():
        observed = values(path)
        summary[f"median_{label}"] = statistics.median(observed)
        summary[f"min_{label}"] = min(observed)
        summary[f"max_{label}"] = max(observed)
    return summary

d06_summaries = [summarize_codec(codec) for codec in ("snappy", "zstd")]
d06_payload = {
    "run_id": run_id,
    "decision": "T01-D06",
    "decision_status": "PENDING_EMPIRICAL_EVIDENCE",
    "winner_selected": False,
    "rows": EXPECTED_ROWS,
    "repetitions_requested": REPETITIONS,
    "writer_settings_held_constant": writer_settings,
    "results": d06_results,
    "summaries": d06_summaries,
}
if REPETITIONS > 0:
    write_json(AUDIT_DIR / "t01_parquet_compression_benchmark.json", d06_payload)
else:
    display(Markdown("**D06 execution:** skipped by the predeclared D04 full-confirmation mode; existing D06 evidence and owner-pending status are unchanged."))
pd.DataFrame(d06_summaries)

**D06 execution:** skipped by the predeclared D04 full-confirmation mode; existing D06 evidence and owner-pending status are unchanged.

,codec,completed_repetitions
0,snappy,0
1,zstd,0


## Evidence closure and interpretation boundary

The final cells validate artifact presence and summarize observations. They deliberately leave T01-D04 and T01-D06 unresolved. Practical thresholds and the codec choice require owner review after this evidence run.

In [5]:
run_config = {
    "run_id": run_id,
    "purpose": "T01_D04_CONTROLLED_FULL_MEMORY_CONFIRMATION_ONLY",
    "execution_mode": EXECUTION_MODE,
    "raw_csv_sha256": raw_hash,
    "source_parquet_sha256": parquet_hash,
    "source_reconciliation_required": True,
    "d04": {"scales": SCALES, "engine": "Pandas + PyArrow", "threshold_predeclared": False},
    "d06": {"executed": False, "reason": "out of scope for controlled D04 confirmation", "winner_selected": False},
    "model_training": False,
    "held_out_evaluation": False,
}
write_json(AUDIT_DIR / "run_config.json", run_config)

if TEMP_DIR.exists() and not any(TEMP_DIR.iterdir()):
    TEMP_DIR.rmdir()

artifact_entries = []
for artifact_path in sorted(AUDIT_DIR.glob("*.json")):
    if artifact_path.name == "artifact_manifest.json":
        continue
    artifact_entries.append({
        "path": artifact_path.relative_to(RUN_ROOT).as_posix(),
        "size_bytes": artifact_path.stat().st_size,
        "sha256": sha256_file(artifact_path),
        "stage": "development_decision_evidence",
    })
artifact_manifest = {
    "run_id": run_id,
    "status": "COMPLETED_WITHOUT_DECISION",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "artifacts": artifact_entries,
}
write_json(AUDIT_DIR / "artifact_manifest.json", artifact_manifest)

resource_frame = pd.DataFrame(d04_results)
compression_frame = pd.DataFrame(d06_summaries)
failures = [r for r in d04_results if r.get("status") != "COMPLETED"] + [r for r in d06_results if r.get("status") != "COMPLETED"]
display(Markdown(
    f"**Evidence run complete:** `{run_id}`  \n"
    f"**D04 full confirmation:** {sum(resource_frame['status'] == 'COMPLETED')}/{len(SCALES)} completed  \n"
    "**D06:** not executed in this confirmation run  \n"
    f"**Failures:** {len(failures)}  \n"
    "**Decision status:** T01-D04 and T01-D06 remain `PENDING_EMPIRICAL_EVIDENCE`; no threshold or winner was selected."
))
display(resource_frame[["requested_rows", "status", "wall_seconds", "peak_rss_bytes", "peak_rss_delta_bytes", "swap_used_increase_bytes"]])
display(compression_frame)
if failures:
    display(failures)

**Evidence run complete:** `t01_d04_confirmation_20260812T043244Z_565265`  
**D04 full confirmation:** 1/1 completed  
**D06:** not executed in this confirmation run  
**Failures:** 0  
**Decision status:** T01-D04 and T01-D06 remain `PENDING_EMPIRICAL_EVIDENCE`; no threshold or winner was selected.

,requested_rows,status,wall_seconds,peak_rss_bytes,peak_rss_delta_bytes,swap_used_increase_bytes
0,13979592,COMPLETED,4.677462,2359005184,2204852224,1207156736


,codec,completed_repetitions
0,snappy,0
1,zstd,0


## Owner decision closure — 2026-08-12

The executed cells above are historical decision evidence and correctly retain the pre-decision `PENDING_EMPIRICAL_EVIDENCE` status. After reviewing that evidence, the owner accepted T01-D01 through T01-D06 in `docs/adr/ADR-T01-data-engineering.md`. The generated JSON and cell outputs are not rewritten retroactively.

- **T01-D04 — ACCEPTED:** use PyArrow Dataset/Scanner projection and batching by default; full Pandas materialization is explicit and operation-specific. There is no universal fixed RAM-percentage threshold. A resource gate fails on OOM/process termination, incorrect or incomplete execution, sustained severe system-memory/pagefile pressure that makes the declared environment unsuitable, or violation of a separately declared operational budget. Process RSS alone is not proof of machine safety.
- **T01-D06 — ACCEPTED:** use ZSTD and retain the benchmarked `1,048,576`-row-group layout. In this bounded workload, all semantic checks passed; ZSTD was about 27% smaller, had materially lower median repeated-read time, and had about 7% higher median write time than Snappy. This is project-specific evidence, not a universal performance claim.

This closure completes the T01 Tutor/decision phase only. It does not implement the production pipeline, train a model, construct a split, access held-out evaluation, or mark Issue #2 complete.